In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [18]:
# =========================================================
# LOAD DATA
# =========================================================

df = pd.read_csv("telco_cleaned.csv")

display(df.head())

print("Dataset shape:", df.shape)

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.50,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,Yes,1,89,5340,Competitor had better devices


Dataset shape: (7043, 33)


In [19]:
# =========================================================
# DEFINE REGRESSION TARGET
# =========================================================
# CLTV proxy:
# If Total Revenue exists, use it.
# Otherwise use Total Charges.

if "Total Revenue" in df.columns:
    target = "Total Revenue"

elif "Total Charges" in df.columns:
    target = "Total Charges"

else:
    raise ValueError("No revenue/charges column found.")


df[target] = pd.to_numeric(
    df[target],
    errors="coerce"
)

df = df.dropna(
    subset=[target]
).copy()

print("Regression target:", target)

display(
    df[[target]].describe()
)

Regression target: Total Charges


,Total Charges
count,7043.000000
mean,2281.916928
std,2265.270398
min,18.800000
25%,402.225000
50%,1397.475000
75%,3786.600000
max,8684.800000


In [20]:
# =========================================================
# REGRESSION FEATURES
# =========================================================
# IMPORTANT:
# Only use features that will also be available in the
# Streamlit customer input form.
#
# We intentionally do NOT use:
# CLTV
# Churn Label
# Churn Value
# Churn Score
# Churn Reason
# Total Charges
# Customer ID
#
# This avoids data leakage and keeps the model compatible
# with the Streamlit application.

regression_features = [
    "Gender",
    "Senior Citizen",
    "Partner",
    "Dependents",
    "Tenure Months",
    "Phone Service",
    "Multiple Lines",
    "Internet Service",
    "Online Security",
    "Online Backup",
    "Device Protection",
    "Tech Support",
    "Streaming TV",
    "Streaming Movies",
    "Contract",
    "Paperless Billing",
    "Payment Method",
    "Monthly Charges"
]


# Check that all required features exist
missing_features = [
    col for col in regression_features
    if col not in df.columns
]

if missing_features:
    raise ValueError(
        f"Missing regression features: {missing_features}"
    )


X = df[regression_features].copy()
y = df[target].copy()


print("Target:", target)
print("X shape:", X.shape)
print("Number of input features:", X.shape[1])
print("Features:")
print(X.columns.tolist())

Target: Total Charges
X shape: (7043, 18)
Number of input features: 18
Features:
['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges']


In [21]:
# =========================================================
# IDENTIFY NUMERIC / CATEGORICAL COLUMNS
# =========================================================

num_cols = X.select_dtypes(
    include=np.number
).columns.tolist()

cat_cols = X.select_dtypes(
    exclude=np.number
).columns.tolist()


print("\nNumeric columns:")
print(num_cols)

print("\nCategorical columns:")
print(cat_cols)


Numeric columns:
['Tenure Months', 'Monthly Charges']

Categorical columns:
['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method']


In [22]:

# =========================================================
# PREPROCESSING
# =========================================================

preprocessor = ColumnTransformer(

    transformers=[

        (
            "num",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="median")
                ),
                (
                    "scaler",
                    StandardScaler()
                )
            ]),
            num_cols
        ),

        (
            "cat",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent")
                ),
                (
                    "onehot",
                    OneHotEncoder(
                        handle_unknown="ignore"
                    )
                )
            ]),
            cat_cols
        )
    ]
)

In [23]:
# =========================================================
# TRAIN / TEST SPLIT
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


print("\nTraining samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


Training samples: 5634
Testing samples: 1409


In [24]:
# =========================================================
# MODELS
# =========================================================

models = {

    "Linear Regression":
        LinearRegression(),

    "Ridge":
        Ridge(alpha=1.0),

    "Random Forest":
        RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        ),

    "Gradient Boosting":
        GradientBoostingRegressor(
            random_state=42
        )
}

In [25]:
# =========================================================
# TRAIN AND EVALUATE
# =========================================================

results = []
fitted_models = {}


for name, model in models.items():

    print("\n" + "=" * 50)
    print("Training:", name)
    print("=" * 50)

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipe.fit(
        X_train,
        y_train
    )

    pred = pipe.predict(
        X_test
    )

    mae = mean_absolute_error(
        y_test,
        pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            pred
        )
    )

    r2 = r2_score(
        y_test,
        pred
    )

    results.append({

        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2

    })

    fitted_models[name] = pipe

    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R2   : {r2:.4f}")


Training: Linear Regression
MAE  : 556.1049
RMSE : 688.9400
R2   : 0.9086

Training: Ridge
MAE  : 556.1622
RMSE : 688.9632
R2   : 0.9086

Training: Random Forest
MAE  : 55.2596
RMSE : 81.2269
R2   : 0.9987

Training: Gradient Boosting
MAE  : 71.9480
RMSE : 99.8849
R2   : 0.9981


In [26]:
# =========================================================
# MODEL COMPARISON
# =========================================================

results_df = pd.DataFrame(
    results
).sort_values(
    "RMSE"
).reset_index(
    drop=True
)


print("\n================ MODEL RESULTS ================")

display(results_df)


================ MODEL RESULTS ================


,Model,MAE,RMSE,R2
0,Random Forest,55.259582,81.226922,0.998730
1,Gradient Boosting,71.948024,99.884884,0.998079
2,Linear Regression,556.104870,688.939993,0.908634
3,Ridge,556.162189,688.963222,0.908628


In [27]:
# =========================================================
# SELECT BEST MODEL
# =========================================================

best_name = results_df.iloc[0]["Model"]

best_model = fitted_models[best_name]

pred = best_model.predict(
    X_test
)

print("\nBest model:", best_name)


Best model: Random Forest


In [14]:
# =========================================================
# SAVE BEST MODEL
# =========================================================

model_path = "cltv_model.pkl"

joblib.dump(
    best_model,
    model_path
)

print(
    f"\n{model_path} saved successfully!"
)


cltv_model.pkl saved successfully!


In [15]:
# =========================================================
# VERIFY SAVED MODEL
# =========================================================

loaded_model = joblib.load(
    model_path
)

print("\nModel:", type(loaded_model))


# A Pipeline does not have n_features_in_ corresponding to
# the original customer inputs in the same way as a fitted
# estimator. The preprocessor stores the original feature names.

saved_preprocessor = loaded_model.named_steps[
    "preprocessor"
]

print(
    "Number of original input features:",
    len(saved_preprocessor.feature_names_in_)
)

print("\nOriginal input feature names:")

for i, feature in enumerate(
    saved_preprocessor.feature_names_in_,
    start=1
):
    print(i, feature)




Model: <class 'sklearn.pipeline.Pipeline'>
Number of original input features: 18

Original input feature names:
1 Gender
2 Senior Citizen
3 Partner
4 Dependents
5 Tenure Months
6 Phone Service
7 Multiple Lines
8 Internet Service
9 Online Security
10 Online Backup
11 Device Protection
12 Tech Support
13 Streaming TV
14 Streaming Movies
15 Contract
16 Paperless Billing
17 Payment Method
18 Monthly Charges


In [16]:
# =========================================================
# FINAL FILE CHECK
# =========================================================

import os

print(
    "\nFile exists:",
    os.path.exists(model_path)
)

if os.path.exists(model_path):

    print(
        "File size:",
        f"{os.path.getsize(model_path):,}",
        "bytes"
    )


print("\nCLTV model training and verification completed successfully!")


File exists: True
File size: 90,369,875 bytes

CLTV model training and verification completed successfully!
